# Biohub - Cell Tracking During Development: Learned Graph w Gap Recovery (学習用解説付き写し)

- **コンペ**: [Biohub - Cell Tracking During Development](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development)（ゼブラフィッシュ胚発生の3D顕微鏡動画から細胞を検出・追跡する、賞金$60,000のResearch Code Competition）
- **元notebook**: [Biohub Cell Tracking: Learned Graph w Gap Recovery](https://www.kaggle.com/code/pilkwang/biohub-cell-tracking-learned-graph-w-gap-recovery) by **Pilkwang Kim**（184 upvotes, Gold, Public Score 0.893 / Best Score 0.894 V32）
- **手法の概要**: `TemporalUNet3D`で細胞中心を検出し、ノード間クロスアテンション transformer で隣接フレーム間のリンク確率（エッジスコア）を学習、ILP(整数計画法)でスパースな追跡グラフを選択したのち、決定的なルールベース後処理（物理motionによる再リンク、有限のgap repair、保守的な分裂復元、track形状のline-fit平滑化）を適用する、学習ベース＋ルールベースのハイブリッド手法。
- **このノートブックについて**: これは学習目的で作成した解説付きの写しです。コード自体は元notebookの内容をほぼそのまま保持していますが、実行はしておらず出力（実行結果）は含みません。分量の都合上、パッケージインストールや環境準備などの定型的なボイラープレートコードは一部要約しています。


## 評価指標について

このコンペのタスクは、3D+時間の顕微鏡動画（ゼブラフィッシュ胚の細胞）から、各フレームの細胞中心（ノード）を検出し、フレーム間で同一細胞を結ぶリンク（エッジ）、および細胞分裂（1つのノードから2つのノードへの分岐）を予測することです。

評価指標は次の式で定義されます。

```
score = adjusted_edge_jaccard + 0.1 * division_jaccard
```

- **adjusted_edge_jaccard**: 予測したエッジ集合と正解エッジ集合のJaccard係数（重なり／和集合）で、ノード同士の追跡リンクがどれだけ正しく張れているかを測ります。
- **division_jaccard**: 細胞分裂イベントの検出精度をJaccard係数で測ったもので、0.1倍の重みで加算されます。分裂は全体のごく一部のイベントですが、無視できない配点になっています。

**なぜこの指標が採用されているか**: 細胞追跡タスクでは「個々の検出がどれだけ正確か」より「時間を通じてどれだけ正しく同一細胞を追跡できたか（リンクの正しさ）」が本質的に重要なので、ノード単体のスコアではなくエッジ（リンク）のJaccard係数を主指標にしています。また細胞分裂は発生生物学的に重要なイベントであるため、別枠で少量ながら加点する設計になっています。

**この notebook の手法が指標とどう整合しているか**: エッジ予測モデル（cross-attention transformer）は学習時から「エッジロス」を直接最適化しており、推論時のILPグラフ構築もエッジ確率を直接使っています。一方この特定のバージョン（V32, "Recall With Cleaner Repair"）は高いrecallの検出閾値を維持しつつ、後処理側（gap close・division復元の閾値）は保守的に絞ることで、誤検出よりも取りこぼしを減らす方向にチューニングされています。


## パイプライン全体像

ノートブック冒頭の著者コメントによると、このモデルラインは以下の学習済みトラッキングパイプラインで構成されています。

- **Center model**: `TemporalUNet3D` が短い時間窓の3Dボリュームから細胞中心のロジット（尤度）マップを予測する（What）。3D UNetは医用画像・顕微鏡画像のセグメンテーションで標準的に使われるアーキテクチャで、エンコーダ・デコーダ構造でボクセル単位の予測を行う（Why UNet）。
- **Edge model**: ノード間クロスアテンションtransformerが、隣接フレーム間の親子候補リンクのスコアを学習する（What）。Attention機構を使うことで、単純な距離だけでなく特徴量の類似性も考慮したリンクスコアを学習できる（Why）。
- **Graph optimizer**: ILP（整数線形計画法）が学習済みエッジスコアとイベントコストから、スパースな追跡グラフを選択する（What）。ILPを使うことで、局所的な貪欲マッチングでは起きがちな矛盾（1つの細胞が2つの子と2つの親を同時に持つ等）をグローバルな制約として排除できる（Why）。
- **Deterministic repair**: 物理的なmotionに基づく再リンク、範囲が限定されたgap repair（検出漏れフレームの補完）、保守的な分裂イベント復元、track形状のline-fit平滑化、という4つのルールベース後処理を行う（What）。学習モデルだけでは検出漏れや細胞の一瞬の消失に対応しきれないため、決定的なルールで補完することで頑健性を上げる（Why）。

このバージョン（"Recall With Cleaner Repair"）は、高いrecallの検出閾値を維持しつつ、gap追加やdivision復元の閾値を締めることで、後処理による過剰な修正（over-repair）が本当に損失の原因なのかを検証するプロファイルです。


## 学習時の目的関数（Training Objective）

著者は数式でモデルの学習方法を説明しています。以下、要点を日本語で解説します。

**検出（Center detection）**: 短い時間窓の3Dボリューム $V_{i:t+W-1}$ を `TemporalUNet3D` エンコーダ $h_\theta$ でエンコードし、$1\times1\times1$ 畳み込み $q_\phi$ でフレームごとの中心ロジット場に変換します。

$$F_t = h_\theta(V_{t:t+W-1}), \quad a_t(\mathbf{r}) = q_\phi(F_t(\mathbf{r})), \quad p_t(\mathbf{r}) = \sigma(a_t(\mathbf{r}))$$

正解（ground truth）のボクセルは正例、それ以外は弱い負例として扱われます。正例・負例それぞれの重みは以下のように別々に正規化されます（クラス不均衡対策、What/Why）。

$$w_+(b) = \frac{1}{N_+(b)}, \quad w_-(b) = \frac{\alpha}{N_-(b)}, \quad \alpha = 0.01$$

検出ロス（$\mathcal{L}_{det}$）は重み付きの binary cross entropy（BCEロジット版）です。

**エッジ（リンク）予測**: 各フレームの局所極大点が候補検出となり、隣接フレームのノード特徴量がbidirectionalクロスアテンションtransformerに入力されます。ソースノード $i$ からターゲットノード $j$ へのエッジロジットは、両ノードの特徴量と相対位置ベクトルから計算されます。学習時はターゲットごとにソフトマックス正規化してエッジ確率 $P_{ij}$ を求め、これにより1つのターゲットに複数の親候補が競合する状況を学習に反映しています（複数の娘への分裂は許容しつつ、同一ターゲットへの多数の親は抑制する設計）。

エッジロスはfocal-styleの重み付きBCEで、正解エッジ・不正解エッジの両方にタッチする行・列のみに限定して計算されます（アノテーションが疎であるため）。

$$\mathcal{L} = \mathcal{L}_{edge} + \lambda_{det}\mathcal{L}_{det}, \quad \lambda_{det}=1$$

**なぜこの設計か**: 検出とリンクを別々の軽量なヘッド（1x1x1畳み込み／クロスアテンション）で学習することで、3D UNetの重い計算を1回のフォワードで使い回しつつ、リンクタスク特有の「競合する候補の中から正しい対応を選ぶ」という性質をソフトマックス正規化で表現しています。


## 推論時のグラフ構築（Submission Graph Construction）とルールベース後処理の数式

推論時は、確率0.985 (`τ`) を超える局所極大点を検出とし、学習済みエッジ予測器から得た候補リンクをILPで解きます（`RUN_CONFIG` の設定に従う）。

**距離の単位変換**: すべての幾何計算はマイクロメートル(µm)単位で行われます（ボクセルサイズ 1.625 x 0.40625 x 0.40625 µm）。

$$d_{\mu m}(i,j) = \sqrt{(1.625\Delta z)^2 + (0.40625\Delta y)^2 + (0.40625\Delta x)^2}$$

**Motion relinker（物理motionによる再リンク）**: 検出済みノードから1ステップ分の時間リンクを再構築します。あるノードに前フレームでの対応（predecessor）があれば、次の位置を外挿します（What: 等速直線運動を仮定した予測）。

$$\hat{\mathbf{r}}_{i,t+1} = \mathbf{r}_{i,t} + \lambda_v(\mathbf{r}_{i,t} - \mathbf{r}_{i,t-1})$$

この予測位置と実際の候補位置との距離をコストとしてHungarian法（`scipy.optimize.linear_sum_assignment`、二部グラフの最小コスト完全マッチングを多項式時間で解くアルゴリズム）で解きます。

$$C_{ij} = d_{motion}(i,j) + 0.05\, d_{raw}(i,j) - \beta P_{ij}^{learned}$$

このnotebookでは $\lambda_v=0.52$, $\beta=0.78$, $R_{tight}=6.2\mu m$, $R_{relaxed}=10.4\mu m$ というパラメータを使用しています（Why: 学習済みエッジ確率を負の項としてコストに加えることで、モデルが自信を持っているリンクほど選ばれやすくなる）。

**Gap repair（検出漏れフレームの補完）**: 1フレーム分の検出漏れであれば、track終端 $t$ とtrack始端 $t+2$ が十分近ければ中間ノードを挿入・再利用します。

$$d_{\mu m}(i,j) \le 2g, \quad g = 5.9\mu m$$

2フレーム分の欠落復元（gap2）はより小さい上限で制御されます（What: 誤って離れた別細胞同士を繋いでしまうリスクを抑えるため、欠落フレーム数が増えるほど許容距離を厳しくする設計）。

**Line-fit smoothing（track形状の平滑化）**: グラフのトポロジー（どのノードがどのノードと繋がっているか）は変えず、座標のみを局所的な線形フィットで滑らかにします。

$$\mathbf{r}_i' = (1-w)\mathbf{r}_i + w\,\mathrm{LineFit}_{N(i)}(t_i), \quad w=0.74$$


In [ ]:
RUN_CONFIG = {
    "allow_artifact_fallback": False,
    "allow_pip_install": False,
    "det_threshold": 0.985,
    "div_drop_to_single_if_bad": True,
    "div_parent_max_um": 10.5,
    "div_sister_max_um": 8.0,
    "experiment_tag": "candidate_20_200ep_recall_clean_repair",
    "gap2_frame_frac_cap": 0.004,
    "gap2_max_links_abs": 120,
    "gap2_max_links_frac": 0.0026,
    "gap2_max_step_um": 4.0,
    "gap2_max_total_um": 9.5,
    "gap2_require_context": True,
    "gap_close_effective_max_gap": 1,
    "gap_close_max_added_abs": 1900,
    "gap_close_max_added_frac": 0.045,
    "gap_close_max_gap": 1,
    "gap_close_reuse_existing": True,
    "gap_close_reuse_um": 3.1,
    "gap_close_um": 5.9,
    "motion_relink_relaxed_um": 10.0,
    "motion_relink_tight_um": 5.95,
    "motion_relink_velocity_weight": 0.5,
    "output_division_geometry_filter": False,
    "output_edge_max_um": 14.2,
    "output_enforce_next_frame": True,
    "output_filter_short_tracks": False,
    "output_gap2_recovery": True,
    "output_gap_close": True,
    "output_keep_division_components": True,
    "output_linefit_smooth": True,
    "output_linefit_weight": 0.74,
    "output_linefit_window": 2,
    "output_min_track_len": 4,
    "output_motion_relink": True,
    "output_prune_isolated": True,
    "output_safe_divisions": True,
    "output_single_child_repair": False,
    "output_single_parent_repair": True,
    "primary_artifact_manifest": "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json",
    "safe_div_existing_child_max_um": 7.4,
    "safe_div_frame_frac_cap": 0.008,
    "safe_div_global_frac_cap": 0.0036,
    "safe_div_max_um": 4.6,
    "safe_div_sister_max_um": 6.9,
    "slice": "",
    "target_artifact_slug": "biohub-tracking-support-pack-50ep-v1",
    "unet_batch_size": 4,
    "use_ilp": True,
    "weights": "weights/unet_transformer/split_0/edge_predictor_best.pth",
}

**このコードは何をしているか（What）**: `RUN_CONFIG` は推論時のあらゆる閾値・パラメータを一箇所にまとめた設定辞書です。検出閾値、gap repairの各種上限（絶対数・全体に対する割合の両方でキャップ）、分裂復元の距離上限、line-fit平滑化の重みなどが含まれます。

**なぜこの設計か（Why）**: 「count-based cap」（`*_max_added_abs`）と「frac-based cap」（`*_max_added_frac`）の両方を同時に設けているのがポイントです。動画1本あたりのノード数はデータセットごとに大きく異なるため、絶対数だけの上限だとノードが少ない動画では効きすぎ、多い動画では効かなすぎる問題が起きます。全体に対する割合の上限を併用することで、どのサイズの動画に対しても「後処理で追加できるエッジ数」が相対的に一定範囲に収まるよう制御しています。


## Artifact and Dependency Setup（環境準備・依存パッケージ解決）

このnotebookは学習済み重みと推論用ソースコードをひとまとめにした「サポートアーティファクト」を添付データセットとして読み込みます。主要な添付パスは `/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json` です。

セットアップは（1）既にインストール済みのモジュールを優先、（2）添付されたwheelファイルを次に試す、（3）明示的に有効化しない限りインターネット経由のインストールは行わない、という順序で依存解決を行います（Code Competitionはインターネットアクセスが無効化されるため、事前にwheelを同梱しておく必要があります）。

以下は依存解決のコードの抜粋です（パッケージ仕様の列挙やアーティファクト検索など定型的な部分が多いため、代表的な関数のみ抜粋しています）。


In [ ]:
import re

os.environ.setdefault("POLARS_PREFER_PKG", "32")

PACKAGE_SPECS = {
    "tracksdata": ("tracksdata", "tracksdata"),
    "zarr": ("zarr", "zarr>=3.0.10,<4"),
    "pyscipopt": ("pyscipopt", "pyscipopt"),
    "geff": ("geff", "geff>=1.3.1.1"),
    "geff_spec": ("geff_spec", "geff-spec<1.2"),
    "ilpy": ("ilpy", "ilpy>=0.5.1"),
    "polars": ("polars", "polars>=1.36"),
    "blosc2": ("blosc2", "blosc2"),
    "dask": ("dask", "dask"),
    "imagecodecs": ("imagecodecs", "imagecodecs"),
    "skimage": ("skimage", "scikit-image>=0.24"),
    "pyarrow": ("pyarrow", "pyarrow"),
    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),
    # ...(以下、他の依存パッケージも同様に列挙。紙面の都合上省略)...
}

# find_offline_package_dirs: /kaggle/working, /kaggle/input 以下を再帰的に走査し、
# *.whl / *.tar.gz / *.zip を含むディレクトリを列挙する(オフラインpipインストール用)。
# purge_imported_modules: パッケージのバージョン競合を避けるため、
# 再インストール前にsys.modulesから該当パッケージを明示的に取り除く。
# polars_runtime_ready: polarsの特定の内部APIが期待通り動くかをランタイムチェックする
# (バージョン差異による無言の不具合を早期検知するための防御的コード)。

**このコードは何をしているか（What）**: Kaggleの Code Competition 環境ではインターネット接続が無効化されるため、必要なライブラリ（`tracksdata`, `zarr`, `polars`, ILP用の `pyscipopt`/`ilpy` など）を事前にダウンロードして添付データセットに同梱し、それをオフラインでインストールする仕組みを実装しています。

**なぜこの設計か（Why）**: Kaggle Notebooksのベース環境には無い、または古いバージョンしか入っていないパッケージを確実に使うため、（1）まず既にインストール済みか確認、（2）無ければ添付wheelから探す、（3）添付にも無ければ最終手段としてインターネットインストールを試みる（ただし既定では無効）、という段階的なフォールバックを組んでいます。これにより再現性を保ちながら、環境差異によるエラーを防いでいます。


## Predict Candidate Graphs（推論の実行）

推論ステップでは、テスト動画1本につき1つの `.geff` グラフファイル（node-link形式のグラフをZarr/HDF5ベースで保存するフォーマット）を書き出します。グラフ予測とCSV変換を分離することで、後処理のデバッグや診断がしやすくなっています。

実際のモデル本体（`TemporalUNet3D` や cross-attention transformer の層定義）はこのnotebook自体には含まれておらず、添付の推論用リポジトリ（`tracking_repo`）内の `scripts/predict_unet_transformer.py` に実装されています。notebook側はこのスクリプトをコマンドライン引数付きでサブプロセス実行するオーケストレーション役に徹しています。


In [ ]:
def list_test_stems() -> list[str]:
    if not TEST_DIR.exists():
        raise FileNotFoundError(f"Test directory does not exist: {TEST_DIR}")
    stems = sorted(path.name[:-5] for path in TEST_DIR.iterdir() if path.name.endswith(".zarr"))
    if not stems:
        raise FileNotFoundError(f"No test .zarr files found in {TEST_DIR}")
    return stems


test_stems = list_test_stems()
print(f"Found {len(test_stems)} test videos")
print(test_stems[:10])

splits_path = REPO_DIR / "kaggle_test_splits_50ep.json"
splits_path.parent.mkdir(parents=True, exist_ok=True)
splits_path.write_text(json.dumps([{"split": 0, "train": [], "test": test_stems}], indent=2))

predict_cmd = [
    sys.executable,
    "scripts/predict_unet_transformer.py",
    "--data-dir",
    str(TEST_DIR),
    "--splits",
    str(splits_path.name),
    "--split",
    "0",
    "--weights",
    WEIGHTS_RELATIVE,
    "--unet-batch-size",
    str(UNET_BATCH_SIZE),
    "--det-threshold",
    str(DET_THRESHOLD),
    "--ilp-edge-weight",
    str(ILP_EDGE_WEIGHT),
    "--ilp-appearance-weight",
    str(ILP_APPEARANCE_WEIGHT),
    "--ilp-disappearance-weight",
    str(ILP_DISAPPEARANCE_WEIGHT),
    "--ilp-division-weight",
    str(ILP_DIVISION_WEIGHT),
]
if USE_ILP:
    predict_cmd.append("--use-ilp")
if SLICE:
    predict_cmd.extend(["--slice", SLICE])

start_time = time.time()
print(" ".join(predict_cmd))
subprocess.run(predict_cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": "src"}, check=True)
predict_seconds = time.time() - start_time

**このコードは何をしているか（What）**: テストセット内の `.zarr` 動画ファイル（Zarr形式は大規模な多次元配列をチャンク分割して保存できるフォーマットで、3D+時間の顕微鏡データによく使われます）を列挙し、外部の推論スクリプトを `subprocess.run` で呼び出しています。`RUN_CONFIG` の各パラメータ（検出閾値、ILPの重み等）はすべてコマンドライン引数として渡されます。

**なぜこの設計か（Why）**: モデル本体のコードをnotebookから切り離し、バージョン管理されたリポジトリ（`tracking_repo`）として扱うことで、（1）notebook自体を短く保ち著者の意図（パラメータ選択・後処理ロジック）を強調できる、（2）モデルコードを他のnotebook・実験間で使い回せる、というメリットがあります。実際の出力ログには「Fold 0: 4 datasets | weights=... | device=cuda | window_size=2 | pool_kernel_um=3.0」「Prediction completed in 6.47 minutes」といった情報が記録されています。


## Build submission.csv（後処理とCSV変換）

推論で得られたグラフに対し、gap close・motion relink・division復元・line-fit smoothingといった決定的な後処理を適用し、コンペ指定フォーマットの `submission.csv` を**メモリに載せず1行ずつストリーミングでディスクに書き出す**関数群です。

標準のgap closer（1フレーム分の欠落のみ対応）と、より厳しい制約のgap2パス（2フレーム分の欠落に対応）を分けているのは、「ゆるい設定を1箇所変えただけで、連続していない（non-consecutive）エッジが紛れ込んでしまう」事故を防ぐためです。


In [ ]:
import tracksdata as td
import numpy as np
import blosc2
from scipy.optimize import linear_sum_assignment

SUBMISSION_COLUMNS = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
CSV_COLUMNS = ["id", *SUBMISSION_COLUMNS]
VOXEL_SCALE_UM = (1.625, 0.40625, 0.40625)


def graph_from_geff(path: Path):
    graph = td.graph.IndexedRXGraph.from_geff(path)
    return graph[0] if isinstance(graph, tuple) else graph


def edge_distance_um(source: dict[str, object], target: dict[str, object]) -> float:
    dz = (float(source["z"]) - float(target["z"])) * VOXEL_SCALE_UM[0]
    dy = (float(source["y"]) - float(target["y"])) * VOXEL_SCALE_UM[1]
    dx = (float(source["x"]) - float(target["x"])) * VOXEL_SCALE_UM[2]
    return math.sqrt(dz * dz + dy * dy + dx * dx)


def point_distance_um(a: tuple[float, float, float], b: tuple[float, float, float]) -> float:
    dz = (a[0] - b[0]) * VOXEL_SCALE_UM[0]
    dy = (a[1] - b[1]) * VOXEL_SCALE_UM[1]
    dx = (a[2] - b[2]) * VOXEL_SCALE_UM[2]
    return math.sqrt(dz * dz + dy * dy + dx * dx)


def node_point(node: dict[str, object]) -> tuple[float, float, float]:
    return (float(node["z"]), float(node["y"]), float(node["x"]))


def edge_sort_key(edge: dict[str, object]) -> tuple[float, float]:
    prob = edge.get("edge_prob")
    prob_value = float(prob) if prob is not None else 0.0
    return prob_value, -float(edge["distance_um"])

# ...(ノードID採番・フレーム読み込みキャッシュなどの補助関数が続く)...

**このコードは何をしているか（What）**: `.geff` グラフをメモリに展開し、ノード間の距離（µm単位）を計算するための共通ユーティリティ関数を定義しています。`edge_sort_key` は「学習済みエッジ確率が高い順、同点なら距離が近い順」で候補を並べ替えるためのキーで、後段のgap close処理でどのペアを優先的に繋ぐかを決めるのに使われます。

**なぜµm単位に統一するのか（Why）**: 顕微鏡のボクセルサイズは軸ごとに異なります（Z軸1.625µm、YX軸0.40625µm）。ボクセル単位のままだと軸によって物理的な意味が違う距離を単純比較してしまう危険があるため、必ず物理単位（µm）に変換してから閾値判定を行っています。


In [ ]:
def gap_close_pass(nodes_by_id, edges, effective_gap_max=1, threshold_um=5.9):
    """1フレーム分の検出漏れを、Hungarian法による最小コスト割当てで補完する。"""
    outgoing = {int(edge["source_id"]) for edge in edges}
    incoming = {int(edge["target_id"]) for edge in edges}
    incident = outgoing | incoming

    ends_by_t: dict[int, list[int]] = {}
    starts_by_t: dict[int, list[int]] = {}
    isolated_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        t = int(node["t"])
        if node_id not in outgoing:
            ends_by_t.setdefault(t, []).append(node_id)
        if node_id not in incoming:
            starts_by_t.setdefault(t, []).append(node_id)
        if node_id not in incident:
            isolated_by_t.setdefault(t, []).append(node_id)

    new_edges: list[dict[str, object]] = []
    used_starts: set[int] = set()

    for gap in range(1, effective_gap_max + 1):
        for t, end_ids in sorted(ends_by_t.items()):
            start_ids = [sid for sid in starts_by_t.get(t + gap + 1, []) if sid not in used_starts]
            if not end_ids or not start_ids:
                continue

            end_points = [node_point(nodes_by_id[eid]) for eid in end_ids]
            start_points = [node_point(nodes_by_id[sid]) for sid in start_ids]
            threshold = threshold_um * (gap + 1)

            d = np.zeros((len(end_ids), len(start_ids)), dtype=np.float64)
            for i, ep in enumerate(end_points):
                for j, sp in enumerate(start_points):
                    d[i, j] = point_distance_um(ep, sp)
            if not np.isfinite(d).any():
                continue

            big = threshold * 1000.0 + 1.0
            cost = np.where(d <= threshold, d, big)
            row_ind, col_ind = linear_sum_assignment(cost)
            for r, c in zip(row_ind, col_ind):
                if d[r, c] > threshold:
                    continue
                source_id = end_ids[int(r)]
                target_id = start_ids[int(c)]
                if source_id in outgoing or target_id in used_starts:
                    continue
                # 中間フレームに合成ノード(synthetic midpoint)を挿入し、
                # refine_synthetic_midpoint で画素強度の重心を使ってサブピクセル精緻化する
                # (詳細は元notebook参照。ここでは骨格のみ抜粋)
                used_starts.add(target_id)
                new_edges.append({"source_id": source_id, "target_id": target_id})

    return new_edges

**このコードは何をしているか（What）**: 上のコードは、gap close処理の中心ロジックを単純化して示したものです。あるフレーム $t$ でtrackが途切れた終端ノード群（`ends_by_t`）と、$t+gap+1$ で新たに現れた始端ノード群（`starts_by_t`）の全ペアの距離行列を作り、Hungarian法（`linear_sum_assignment`）で全体最適なマッチングを1回で解いています。閾値を超えるペアには大きなコスト（`big`）を割り当てることで実質的にマッチング対象から除外しています。

**なぜHungarian法を使うのか（Why）**: 単純に「一番近いペアから貪欲に繋ぐ」方法だと、後から見るとより良いペアが選べたはずなのに早い者勝ちで悪いペアを繋いでしまうことがあります。Hungarian法は全体のコスト合計を最小化する厳密解を多項式時間（$O(n^3)$）で求められるため、局所的な貪欲法より一貫性の高いマッチングが得られます。

**実行結果（著者のログより）**: `Found 4 prediction graphs` → `Wrote /kaggle/working/submission.csv with 267,392 rows`（ノード行 136,864 + エッジ行 130,528）。node/edge統計のサマリーでは、動画あたり平均 division_like_sources（分裂候補）が104件、`edge_to_node比`が0.95前後という値が出力されています。


## この手法から学べる主要テクニック

- Difference of Gaussiansのような古典的画像処理ではなく、`TemporalUNet3D` による学習ベースの中心検出と、cross-attention transformerによる学習ベースのエッジ（リンク）スコアリングを組み合わせている。
- ILP（整数線形計画法）でグラフ全体を大域的に最適化することで、局所的な貪欲マッチングでは防げない矛盾（1つのノードに複数の親、など）を排除している。
- Hungarian法（`linear_sum_assignment`）を使った厳密な最小コストマッチングによるgap close・motion relink。
- 絶対数キャップと割合キャップを併用した、動画サイズに頑健な後処理パラメータ設計。
- 環境非依存性を保つための、多段フォールバック付きオフライン依存パッケージ解決の仕組み。
- サブピクセル精度の中心精緻化（`refine_synthetic_midpoint`）を、画素強度を重みとした重心計算で実現。
